In [26]:
from langgraph.graph import StateGraph, START, END
from IPython.display import Image
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
import operator

In [ ]:
from dotenv import load_dotenv
load_dotenv("../../../LangChainModels/.env") # Load environment variables from .env file

True

In [28]:
model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

Structured Output for LLM

In [29]:
class EvaluationSchema(BaseModel):
    feedback : str = Field(description="detailed feedback for the essay")
    score: int = Field(description="score for the essay out of 10", ge=0, le= 10)

In [30]:
StructuredModel = model.with_structured_output(EvaluationSchema)

In [31]:
class State(TypedDict):
    essay : str
    clarity_of_thought : str
    depth_of_analysis : str
    language_and_style : str
    final_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float

In [32]:
def evaluate_clarity(state: State) -> State:
    prompt = f"Evaluate the clarity of thought in the following essay:\n\n{state['essay']}\n\nProvide detailed feedback and a score out of 10."
    result = StructuredModel.invoke(prompt)
    
    return {
        'clarity_of_thought': result.feedback,
        'individual_scores': [result.score],
    }

In [33]:
def evaluate_depth(state: State) -> State:
    prompt = f"Evaluate the depth of analysis in the following essay:\n\n{state['essay']}\n\nProvide detailed feedback and a score out of 10."
    result = StructuredModel.invoke(prompt)
    
    return {
        'depth_of_analysis': result.feedback,
        'individual_scores': [result.score],
    }

In [34]:
def evaluate_language(state: State) -> State:
    prompt = f"Evaluate the language and style in the following essay:\n\n{state['essay']}\n\nProvide detailed feedback and a score out of 10."
    result = StructuredModel.invoke(prompt)
    
    return {
        'language_and_style': result.feedback,
        'individual_scores': [result.score],
    }

In [35]:
def final_evaluation(state: State) -> State:
    #summary feedback
    prompt = f"Based on the following feedback, provide a final summary feedback for the essay:\n\nClarity of Thought: {state['clarity_of_thought']}\nDepth of Analysis: {state['depth_of_analysis']}\nLanguage and Style: {state['language_and_style']}\n\nProvide a concise summary feedback."
    final_feedback = model.invoke(prompt).content
    avg_score = sum(state['individual_scores']) / len(state['individual_scores'])
    
    
    return {
        'final_feedback': final_feedback,
        'avg_score': avg_score,
    }

In [36]:
graph = StateGraph(State)

In [37]:
graph.add_node('evaluate_clarity', evaluate_clarity)
graph.add_node('evaluate_depth', evaluate_depth)
graph.add_node('evaluate_language', evaluate_language)
graph.add_node('final_evaluation', final_evaluation)


In [38]:
graph.add_edge(START, 'evaluate_clarity')
graph.add_edge(START, 'evaluate_depth')
graph.add_edge(START, 'evaluate_language')
graph.add_edge('evaluate_clarity', 'final_evaluation')
graph.add_edge('evaluate_depth', 'final_evaluation')
graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('final_evaluation', END)

In [39]:
app = graph.compile()

In [40]:
result = app.invoke({'essay': "The essay discusses the impact of climate change on polar bears. It highlights the melting ice caps and the resulting loss of habitat for these animals. The essay also touches upon the broader implications of climate change on global ecosystems and human societies."})

In [41]:
print("Final Feedback:", result)

Final Feedback: {'essay': 'The essay discusses the impact of climate change on polar bears. It highlights the melting ice caps and the resulting loss of habitat for these animals. The essay also touches upon the broader implications of climate change on global ecosystems and human societies.', 'clarity_of_thought': 'The essay presents a clear and focused discussion on the impact of climate change on polar bears, effectively highlighting the critical issue of melting ice caps and habitat loss. However, while it mentions broader implications for global ecosystems and human societies, it lacks depth in exploring these connections. The transitions between ideas could be smoother, and the essay would benefit from more specific examples or data to support its claims. Overall, the clarity of thought is good, but it could be improved with more detailed exploration and stronger connections between points.', 'depth_of_analysis': 'The essay provides a basic overview of the impact of climate chang